In [1]:
import os
import pandas as pd

In [2]:
ROOT_DIR = os.path.dirname(os.getcwd())
QC_CSV_PATH = os.path.join(ROOT_DIR, 'data', 'processed', 'skempi_v2_qc.csv')
SKEMPI_PDBS_DIR = os.path.join(ROOT_DIR, 'data', 'raw', 'SKEMPI2_PDBs', 'PDBs')

In [3]:
print(f"Checking PDB directory: {SKEMPI_PDBS_DIR}")
print(f"Reading QC dataset from: {QC_CSV_PATH}")

Checking PDB directory: /home/maria/projects/BioData-QC/data/raw/SKEMPI2_PDBs/PDBs
Reading QC dataset from: /home/maria/projects/BioData-QC/data/processed/skempi_v2_qc.csv


# Extract PDBs

In [4]:
# Load dataset and extract unique 4-character PDB IDs
df = pd.read_csv(QC_CSV_PATH)

# SKEMPI's '#Pdb' column uses formats like '1CSE_E_I' or '1CSE'
df['pdb_code'] = df['#Pdb'].astype(str).str[:4].str.upper()
required_pdbs = set(df['pdb_code'].unique())

print(f"Total rows in dataset: {len(df)}")
print(f"Unique PDB codes required: {len(required_pdbs)}")

Total rows in dataset: 6798
Unique PDB codes required: 341


# Inventory local files

In [5]:
# Inventory Local Files (Ignoring hidden '._' system files)
all_files = os.listdir(SKEMPI_PDBS_DIR)

existing_pdbs = set()
existing_mappings = set()

for fname in all_files:
    # Ignore macOS hidden system files
    if fname.startswith('._'):
        continue
        
    fname_upper = fname.upper()
    if fname_upper.endswith('.PDB') or fname_upper.endswith('.ENT'):
        clean_id = fname_upper.replace('PDB', '').replace('.ENT', '').replace('.PDB', '')[:4]
        existing_pdbs.add(clean_id)
    elif fname_upper.endswith('.MAPPING'):
        clean_id = fname_upper.split('.')[0][:4]
        existing_mappings.add(clean_id)

print(f"Found {len(existing_pdbs)} valid PDB structural files locally.")
print(f"Found {len(existing_mappings)} valid mapping files locally.")

Found 345 valid PDB structural files locally.
Found 345 valid mapping files locally.


# Required PDBs

In [6]:
# Calculate missing files relative to dataset requirements
missing_pdbs = required_pdbs - existing_pdbs
missing_mappings = required_pdbs - existing_mappings

found_pdbs = required_pdbs.intersection(existing_pdbs)
found_mappings = required_pdbs.intersection(existing_mappings)

print("=" * 50)
print("              PDB AUDIT RESULTS SUMMARY            ")
print("=" * 50)
print(f"PDB Structure Coverage : {len(found_pdbs)} / {len(required_pdbs)} ({len(found_pdbs)/len(required_pdbs)*100:.1f}%)")
print(f"Mapping File Coverage  : {len(found_mappings)} / {len(required_pdbs)} ({len(found_mappings)/len(required_pdbs)*100:.1f}%)")
print("=" * 50)

if missing_pdbs:
    print(f"\n[!] Missing {len(missing_pdbs)} PDB files:")
    print(sorted(list(missing_pdbs)))
else:
    print("\n[SUCCESS] All required PDB structural files are present locally!")

if missing_mappings:
    print(f"\n[!] Missing {len(missing_mappings)} mapping files:")
    print(sorted(list(missing_mappings)))
else:
    print("[SUCCESS] All required mapping files are present locally!")

              PDB AUDIT RESULTS SUMMARY            
PDB Structure Coverage : 341 / 341 (100.0%)
Mapping File Coverage  : 341 / 341 (100.0%)

[SUCCESS] All required PDB structural files are present locally!
[SUCCESS] All required mapping files are present locally!


# Check mapping file

In [8]:
# Inspecting a Real Sample .mapping File
sample_pdb = list(found_mappings)[0] if 'found_mappings' in locals() and found_mappings else None

if sample_pdb:
    sample_mapping_path = None
    for fname in os.listdir(SKEMPI_PDBS_DIR):
        # Ensure we skip AppleDouble hidden files
        if not fname.startswith('._') and sample_pdb in fname.upper() and fname.endswith('.mapping'):
            sample_mapping_path = os.path.join(SKEMPI_PDBS_DIR, fname)
            break

    if sample_mapping_path:
        print(f"--- Sample content of {os.path.basename(sample_mapping_path)} ---")
        with open(sample_mapping_path, 'r', encoding='utf-8', errors='ignore') as f:
            lines = [f.readline() for _ in range(15)]
            print("".join(lines))
else:
    print("No valid mapping files available to display.")

--- Sample content of 1FCC.mapping ---
PRO A 238  1
SER A 239  2
VAL A 240  3
PHE A 241  4
LEU A 242  5
PHE A 243  6
PRO A 244  7
PRO A 245  8
LYS A 246  9
PRO A 247  10
LYS A 248  11
ASP A 249  12
THR A 250  13
LEU A 251  14
MET A 252  15

